# Tutorial 3 — AO Projection and Chemical Character Screening

After the DMRG entropy step has pruned the CAS down to the most entangled
orbitals, **AO projection** refines the selection by asking: *how much does
each surviving orbital look like a specific atomic orbital group?*

This notebook covers:
1. How to obtain AO label strings — manually or via **MolTagger**
2. `ProjectionConfig` — all parameters explained
3. Running `select_by_ao_projection` directly
4. Threshold mode (`contribution` + `contribution_type`)
5. Top-N ranking mode (`top_n`)
6. Per-group thresholds and multi-group selection
7. Restricting the search to entropy-screened orbitals
8. Visualising AO contributions

**Prerequisites**: PySCF and NumPy.  `block2` is *not* required for this
notebook — the entropy-screened space is simulated directly.

## 1. Obtaining AO label strings

AO labels follow the PySCF `mol.search_ao_label` convention:

```
"<Element> <shell>"          # e.g. "C 2pz", "Fe 3d", "O 2p"
"<Atom index> <Element> <shell>"  # pin to a specific atom
```

You can define them in two ways:

### Option A — by hand

Inspect the PySCF AO list and write the strings directly:

```python
mol.ao_labels()          # list every AO in the molecule
mol.search_ao_label("C 2pz")  # verify which AOs match
```

### Option B — MolTagger (recommended for large molecules)

**MolTagger** (`aegiss.utils.mol_tagger`) provides an interactive 3D viewer
where you click on atoms and get ready-to-paste `ao_labels` strings:

```python
from aegiss.utils.mol_tagger import MolTagger

tagger = MolTagger(mol)   # pass the pyscf Mole object
tagger.run()              # opens the GUI
```

After selecting the atoms of interest in the viewer, copy the suggested
labels from the **AO Labels** panel.  The tool auto-detects:

| System | Suggested labels |
|---|---|
| Planar π system (benzene) | `"C 2pz"` |
| Non-planar organic (CH₄) | `"C 2p"` with a warning |
| Transition metal (Fe) | `"Fe 3d"` |
| Lanthanide (Ce) | `["Ce 4f", "Ce 5d"]` |
| σ scaffold | `"C 2px"`, `"C 2py"`, `"C 2s"`, `"H 1s"` |

> The labels suggested by MolTagger are passed **directly** to
> `ProjectionConfig(ao_labels=...)` with no further editing needed.

## 2. Imports

In [ ]:
import numpy as np

from aegiss import ProjectionConfig
from aegiss.active_space.ao_projection import select_by_ao_projection
from aegiss.initial_guess.scf import build_mol, run_scf

## 3. Molecule and mean-field setup

We use benzene as the working example: it has a clear π system (`C 2pz`)
that should be captured by AO projection.

In [ ]:
benzene_geom = """
C   1.40272  0.00000  0.00000
C   0.70136  1.21451  0.00000
C  -0.70136  1.21451  0.00000
C  -1.40272  0.00000  0.00000
C  -0.70136 -1.21451  0.00000
C   0.70136 -1.21451  0.00000
H   2.49118  0.00000  0.00000
H   1.24559  2.15720  0.00000
H  -1.24559  2.15720  0.00000
H  -2.49118  0.00000  0.00000
H  -1.24559 -2.15720  0.00000
H   1.24559 -2.15720  0.00000
"""

mol = build_mol(
    atom=benzene_geom,
    basis="ccpvdz",
    symmetry=False,
    verbose=0,
)

mf = run_scf(mol, method="RHF")
print(f"RHF energy  : {mf.e_tot:.8f} Ha")
print(f"AOs         : {mol.nao}")
print(f"MOs         : {mf.mo_coeff.shape[1]}  (occ: {int(np.sum(mf.mo_occ > 0))})")

### Inspect the AO list

Verify that `"C 2pz"` matches the expected AOs before building the config.

In [ ]:
pz_indices = mol.search_ao_label("C 2pz")
print(f"AOs matching 'C 2pz': indices {pz_indices}")
print("Labels:")
for i in pz_indices:
    print(f"  [{i:>2d}]  {mol.ao_labels()[i]}")

## 4. ProjectionConfig

`ProjectionConfig` collects all AO projection settings.

| Field | Type | Default | Meaning |
|---|---|---|---|
| `ao_labels` | `list[str \| list[str]]` | — | AO groups to project onto |
| `basis` | `str \| None` | `'minao'` | Basis for projection (`None` → use molecule basis) |
| `contribution` | `float \| list[float]` | `0.5` | Weight threshold (or per-group list) |
| `contribution_type` | `'a'` or `'r'` | `'a'` | `'a'` = absolute; `'r'` = relative to max weight |
| `top_n` | `int \| None` | `None` | Keep top-N orbitals per group instead of thresholding |
| `ranking` | `'subset'` or `'global'` | `'subset'` | Apply projection per subspace or on combined set |

> **`ao_labels` format**: a flat string selects one group; a nested list
> `["Fe 3d", "Fe 4d"]` treats both shells as a single group (contributions
> are summed).  Groups are processed independently and their results unioned.

In [ ]:
proj_config = ProjectionConfig(
    ao_labels=["C 2pz"],     # the π system of benzene
    basis="minao",
    contribution=0.5,         # keep orbitals with weight >= 0.5 (absolute)
    contribution_type="a",
)
print(proj_config)

## 5. Running AO projection (no entropy pre-screening)

`select_by_ao_projection` can be called without a `restricted_space` argument
to screen directly over all MOs.  This is useful for a quick sanity check.

In [ ]:
active_indices = select_by_ao_projection(
    mf=mf,
    ao_labels=proj_config.ao_labels,
    projected_basis=proj_config.basis,
    contribution=proj_config.contribution,
    contribution_type=proj_config.contribution_type,
    top_n=proj_config.top_n,
)

print(f"Selected orbital indices : {active_indices}")
print(f"Number of active orbitals: {len(active_indices)}")

## 6. Threshold modes: absolute vs relative

The weight $w_p^D = \sum_{\eta \in D} |[\mathbf{O}_E^D]_{\eta p}|$ is an L1
norm of the projected MO coefficients over the AO group $D$.

- **`contribution_type='a'`** (absolute): keep orbital $p$ if $w_p^D \geq \epsilon$.
  Useful when you have a physical intuition for the expected weight scale.

- **`contribution_type='r'`** (relative): normalise all weights by
  $\max_p w_p^D$ and keep orbital $p$ if $w_p^D / \max \geq \epsilon$.
  The threshold is now dimensionless and in [0, 1] — easier to tune across
  different systems.

In [ ]:
print("=== Absolute threshold (contribution_type='a') ===")
for threshold in (0.2, 0.5, 1.0, 1.5):
    idxs = select_by_ao_projection(
        mf=mf,
        ao_labels=["C 2pz"],
        contribution=threshold,
        contribution_type="a",
    )
    print(f"  threshold={threshold:.1f}  →  {len(idxs)} orbital(s): {list(idxs)}")

print()
print("=== Relative threshold (contribution_type='r') ===")
for fraction in (0.1, 0.3, 0.5, 0.8):
    idxs = select_by_ao_projection(
        mf=mf,
        ao_labels=["C 2pz"],
        contribution=fraction,
        contribution_type="r",
    )
    print(f"  fraction={fraction:.1f}   →  {len(idxs)} orbital(s): {list(idxs)}")

## 7. Top-N ranking mode

Instead of a threshold you can request the **top N** orbitals by weight.
This guarantees a fixed number of orbitals per group regardless of the
weight distribution — useful when you know the target CAS size in advance.

In [ ]:
print("=== Top-N selection ===")
for n in (3, 6, 9, 12):
    idxs = select_by_ao_projection(
        mf=mf,
        ao_labels=["C 2pz"],
        top_n=n,
    )
    print(f"  top_n={n:>2d}  →  {len(idxs)} orbital(s): {list(idxs)}")

For benzene with `"C 2pz"` we expect exactly **6 π MOs** (the bonding,
non-bonding, and anti-bonding π orbitals).  `top_n=6` should recover the
canonical π CAS(6e, 6o).

## 8. Multiple AO groups

Pass multiple entries in `ao_labels` to screen against several chemical
characters simultaneously.  The final active space is the **union** of
orbitals selected by each group.

You can supply:
- A single threshold applied to all groups
- A **per-group threshold list** (length must match `ao_labels`)
- A **per-group `top_n` list**

Example: select π carbons **and** the σ C–H bonds.

In [ ]:
# Two groups: π system and C–H sigma bonds
idxs_multi = select_by_ao_projection(
    mf=mf,
    ao_labels=["C 2pz", "H 1s"],
    contribution=[0.5, 0.3],   # per-group absolute thresholds
    contribution_type="a",
)
print(f"π + σ selection: {list(idxs_multi)}  ({len(idxs_multi)} orbitals)")

In [ ]:
# Alternative: top-N per group
idxs_topn = select_by_ao_projection(
    mf=mf,
    ao_labels=["C 2pz", "H 1s"],
    top_n=[6, 3],              # 6 π MOs + 3 σ MOs
)
print(f"π (top 6) + σ (top 3): {list(idxs_topn)}  ({len(idxs_topn)} orbitals)")

### Grouped labels (multi-shell metals)

For transition metals and f-block elements you often need to group
multiple shells into a single AO entry — the contributions are **summed**
before thresholding.  MolTagger does this automatically for lanthanides
and actinides.

```python
# Cerium: treat 4f + 5d as a single group
ao_labels = [["Ce 4f", "Ce 5d"]]

# Iron porphyrin: three independent groups
ao_labels = ["Fe 3d", "N 2p", "C 2pz"]
```

## 9. Restricting to the entropy-screened space

In the full AEGISS pipeline the AO projection step operates *only on the
orbitals that survived entropy screening*.  Pass `restricted_space` as the
`(screened_occ, screened_virt)` tuple returned by `select_by_entropy`.

Here we simulate that step by hand-picking a reduced orbital list.

In [ ]:
from aegiss.utils.orbital_indices import get_subspaces_from_cas

# Simulate: CAS(6e, 6o) — the benzene π space
n_occ = int(np.sum(mf.mo_occ > 0))
occ_orbs, virt_orbs = get_subspaces_from_cas(mf, ncas=6, nelecas=6)
print(f"CAS occupied  : {occ_orbs}")
print(f"CAS virtual   : {virt_orbs}")

# (occ_orbs, virt_orbs) are the format returned by select_by_entropy
idxs_restricted = select_by_ao_projection(
    mf=mf,
    ao_labels=["C 2pz"],
    contribution=0.3,
    contribution_type="r",
    restricted_space=(occ_orbs, virt_orbs),
)
print(f"\nActive orbitals after projection in CAS(6,6): {list(idxs_restricted)}")

### `ranking` parameter

When `restricted_space` is a `(occ, virt)` tuple:

| `ranking` | Behaviour |
|---|---|
| `'subset'` (default) | Threshold applied independently to occupied and virtual subspaces |
| `'global'` | Both subspaces are merged first; single threshold over the combined pool |

`'subset'` is almost always preferable — it avoids the occupied space being
dominated by strongly entangled virtual orbitals (or vice-versa).

## 10. Visualising AO contributions

Pass `save_path` to write a contribution heatmap to disk.  This is
especially useful when debugging multi-group selections for large
transition-metal systems.

In [ ]:
import os
import tempfile

heatmap_path = os.path.join(tempfile.gettempdir(), "ao_contribution.png")

idxs_plot = select_by_ao_projection(
    mf=mf,
    ao_labels=["C 2pz", "H 1s"],
    contribution=0.3,
    contribution_type="r",
    save_path=heatmap_path,
)
print(f"Heatmap written to: {heatmap_path}")

### Direct control: `plot_ao_contribution`

`save_path` uses sensible defaults, but you can call the plotting function
directly for finer control:

```python
from aegiss.utils.plotting import plot_ao_contribution

plot_ao_contribution(
    alpha_contributions,
    beta_contributions,        # optional — unrestricted (UHF) case only
    folder="./",
    filename="ao",
    normalization="global",    # "global" (default) or "relative"
)
```

- **`normalization`** — `"global"` scales the whole heatmap by a single
  maximum, so weaker AO groups stay visibly dim relative to dominant ones.
  `"relative"` scales each group by its own maximum, emphasising within-group
  patterns at the cost of cross-group magnitude comparison.
- **`ao_contributions_beta`** — when supplied (unrestricted calculations),
  each AO group is drawn as an interleaved alpha-above-beta row pair, with a
  thin white line separating consecutive groups. Both spins are normalised
  jointly per group so spin asymmetry stays visible. Because this adds *rows*
  rather than *columns*, it never worsens the horizontal crowding of wide
  orbital ranges.

See the [`plot_ao_contribution`](../../../README.md#plot_ao_contribution)
reference for the full parameter list.

## Summary

- AO label strings follow the PySCF `mol.search_ao_label` format.
  **MolTagger** can generate them automatically from an interactive 3D viewer.
- `ProjectionConfig` centralises all AO projection settings:
  `ao_labels`, `basis`, `contribution`, `contribution_type`, `top_n`, `ranking`.
- `contribution_type='a'` applies an absolute weight cutoff;
  `'r'` normalises by the maximum weight first.
- `top_n` guarantees a fixed number of orbitals per group.
- Per-group thresholds and per-group `top_n` lists allow fine-grained
  control in multi-group selections.
- `restricted_space=(screened_occ, screened_virt)` limits the search to the
  entropy-screened pool from the previous step.

**Next**: [04_aegiss.ipynb](04_aegiss.ipynb) — combining
all four config objects into a single end-to-end AEGISS run.